In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
import ast
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/GaRAGe_UND_Gemini_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_GaRAGe_UND_qa_Gemini_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 603 examples [00:00, 18321.97 examples/s]


In [3]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/GaRAGe_UND_Gemini_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 23.46ba/s]


2819627

## Rewriting with GPT-4o
GPT-4o rewriting, then Gemini QA later

In [4]:
from helper_functions_qr import modification_in_batch, find_failed_rows_simple

In [5]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = "gpt-4o-2024-11-20"
input_file = "./intermediate/GaRAGe_UND_Gemini_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl"


In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Total samples to process: 603
Batch size: 3


Processing batches:   0%|          | 0/201 [00:00<?, ?it/s]

Processing batches:  89%|████████▊ | 178/201 [21:39<02:26,  6.36s/it] 

Error processing sample 536: Invalid \escape: line 3 column 43 (char 137)


Processing batches: 100%|██████████| 201/201 [24:04<00:00,  7.19s/it]


All batch processing completed! Total processed: 603 samples
Results saved to: ./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl


In [7]:
find_failed_rows_simple(input_file, output_file)

=== 查找失败的行（简单方法）===
发现 0 个失败的行:


[]

In [8]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,What strategies can businesses employ to mitig...,What specific strategies can U.S.-based busine...,"[To mitigate ASC 842 compliance challenges, bu...",[* Start early with planning and assessment\...,The query seeks strategies to address ASC 842 ...,0.169312,0.0,0.50
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Convolutional Neural Network...,[The Deep Retinal Convolutional Neural Network...,[* The Deep Retinal Convolution Neural Netwo...,The query references 'Deep Retinal Convolution...,0.193548,0.0,0.00
2,"How does Zoe Law's ""Legends"" exhibition reflec...","How does Zoë Law's ""Legends"" exhibition use in...","[The ""Legends"" exhibition by Zoë Law reflects ...",[* Empowerment of women through reimagined m...,The query specifies a particular artist (Zoe L...,0.178862,0.0,0.00
3,How does the involvement of Tyco Ventures and ...,How does the $25 million investment by Tyco Ve...,[The involvement of Tyco Ventures and Integral...,[* **Technological Prospects:**\n * Tyc...,The query identifies specific entities (Tyco V...,0.321678,0.0,0.75
4,How has the Drake-Kendrick Lamar feud influenc...,How has the ongoing feud between Drake and Ken...,[The feud between Drake and Kendrick Lamar has...,[* Re-emphasis on lyrical prowess and battle...,The query identifies a specific subject (the D...,0.190476,0.0,1.00
...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,How did Elon Musk's opposition to the proposed...,[Elon Musk's opposition led to the president-e...,[There is no widely reported instance of Elon ...,The query lacks critical specificity. It refer...,0.113821,0.0,0.00
599,What challenges does self-managed OpenSearch d...,What are the key challenges faced in self-mana...,[Self-managed OpenSearch deployments face seve...,[* Complex setup and configuration\n* Ongo...,The query asks about challenges associated wit...,0.129870,0.0,1.00
600,What are the implications of Cleveland-Cliffs ...,What are the potential implications of Clevela...,[Cleveland-Cliffs CEO's plan to make another o...,[* Prolonged uncertainty for U.S. Steel's fu...,The query specifies key elements: the subject ...,0.125000,0.0,0.75
601,How has the expansion of telehealth services u...,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,[* Increased access to primary and specialty...,The query specifies the subject matter (telehe...,0.366972,0.0,1.00


## Modified queries QA using Gemini-2.5-Flash

### Loading modified data

In [9]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

# For issue resolution for QA, comment when there is no issue
#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 603 examples [00:00, 29851.46 examples/s]


### Implementation

In [10]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [11]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 61/61 [53:19<00:00, 52.46s/it] 


In [12]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 38.19ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,What strategies can businesses employ to mitig...,What specific strategies can U.S.-based busine...,"[To mitigate ASC 842 compliance challenges, bu...",[* Start early with planning and assessment\...,The query seeks strategies to address ASC 842 ...,0.169312,0,0.50,[U.S.-based businesses can employ the followin...
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Convolutional Neural Network...,[The Deep Retinal Convolutional Neural Network...,[* The Deep Retinal Convolution Neural Netwo...,The query references 'Deep Retinal Convolution...,0.193548,0,0.00,[Here's how the DCNN approach improves speech ...
2,"How does Zoe Law's ""Legends"" exhibition reflec...","How does Zoë Law's ""Legends"" exhibition use in...","[The ""Legends"" exhibition by Zoë Law reflects ...",[* Empowerment of women through reimagined m...,The query specifies a particular artist (Zoe L...,0.178862,0,0.00,[* By featuring influential figures from art...
3,How does the involvement of Tyco Ventures and ...,How does the $25 million investment by Tyco Ve...,[The involvement of Tyco Ventures and Integral...,[* **Technological Prospects:**\n * Tyc...,The query identifies specific entities (Tyco V...,0.321678,0,0.75,[* Accelerates R&D and product development t...
4,How has the Drake-Kendrick Lamar feud influenc...,How has the ongoing feud between Drake and Ken...,[The feud between Drake and Kendrick Lamar has...,[* Re-emphasis on lyrical prowess and battle...,The query identifies a specific subject (the D...,0.190476,0,1.00,[* **Artistic Competition:** It significantl...
...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,How did Elon Musk's opposition to the proposed...,[Elon Musk's opposition led to the president-e...,[There is no widely reported instance of Elon ...,The query lacks critical specificity. It refer...,0.113821,0,0.00,[There is no publicly available information in...
599,What challenges does self-managed OpenSearch d...,What are the key challenges faced in self-mana...,[Self-managed OpenSearch deployments face seve...,[* Complex setup and configuration\n* Ongo...,The query asks about challenges associated wit...,0.129870,0,1.00,[* **Infrastructure Management:**\n * M...
600,What are the implications of Cleveland-Cliffs ...,What are the potential implications of Clevela...,[Cleveland-Cliffs CEO's plan to make another o...,[* Prolonged uncertainty for U.S. Steel's fu...,The query specifies key elements: the subject ...,0.125000,0,0.75,[* Increased consolidation of the domestic s...
601,How has the expansion of telehealth services u...,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,[* Increased access to primary and specialty...,The query specifies the subject matter (telehe...,0.366972,0,1.00,"[* **Service utilization rates:** Increased,..."


## Evaluations

### Squad EM+F1

In [13]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GaRAGe_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 603 examples [00:00, 30160.46 examples/s]


In [14]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_gpt4o_GaRAGe_UND_Gemini_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_gpt4o_GaRAGe_UND_Gemini_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 41.49ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,What strategies can businesses employ to mitig...,What specific strategies can U.S.-based busine...,"[To mitigate ASC 842 compliance challenges, bu...",[* Start early with planning and assessment\...,The query seeks strategies to address ASC 842 ...,0.169312,0,0.50,[U.S.-based businesses can employ the followin...,0,0.257426
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Convolutional Neural Network...,[The Deep Retinal Convolutional Neural Network...,[* The Deep Retinal Convolution Neural Netwo...,The query references 'Deep Retinal Convolution...,0.193548,0,0.00,[Here's how the DCNN approach improves speech ...,0,0.378698
2,"How does Zoe Law's ""Legends"" exhibition reflec...","How does Zoë Law's ""Legends"" exhibition use in...","[The ""Legends"" exhibition by Zoë Law reflects ...",[* Empowerment of women through reimagined m...,The query specifies a particular artist (Zoe L...,0.178862,0,0.00,[* By featuring influential figures from art...,0,0.317757
3,How does the involvement of Tyco Ventures and ...,How does the $25 million investment by Tyco Ve...,[The involvement of Tyco Ventures and Integral...,[* **Technological Prospects:**\n * Tyc...,The query identifies specific entities (Tyco V...,0.321678,0,0.75,[* Accelerates R&D and product development t...,0,0.327869
4,How has the Drake-Kendrick Lamar feud influenc...,How has the ongoing feud between Drake and Ken...,[The feud between Drake and Kendrick Lamar has...,[* Re-emphasis on lyrical prowess and battle...,The query identifies a specific subject (the D...,0.190476,0,1.00,[* **Artistic Competition:** It significantl...,0,0.242424
...,...,...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,How did Elon Musk's opposition to the proposed...,[Elon Musk's opposition led to the president-e...,[There is no widely reported instance of Elon ...,The query lacks critical specificity. It refer...,0.113821,0,0.00,[There is no publicly available information in...,0,0.254417
599,What challenges does self-managed OpenSearch d...,What are the key challenges faced in self-mana...,[Self-managed OpenSearch deployments face seve...,[* Complex setup and configuration\n* Ongo...,The query asks about challenges associated wit...,0.129870,0,1.00,[* **Infrastructure Management:**\n * M...,0,0.339506
600,What are the implications of Cleveland-Cliffs ...,What are the potential implications of Clevela...,[Cleveland-Cliffs CEO's plan to make another o...,[* Prolonged uncertainty for U.S. Steel's fu...,The query specifies key elements: the subject ...,0.125000,0,0.75,[* Increased consolidation of the domestic s...,0,0.194175
601,How has the expansion of telehealth services u...,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,[* Increased access to primary and specialty...,The query specifies the subject matter (telehe...,0.366972,0,1.00,"[* **Service utilization rates:** Increased,...",0,0.319018


In [15]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 0.00
New answers after modification F1 Score (avg): 22.48
Original answers Exact Match (avg): 0.00
Original answers F1 Score (avg): 15.41
F1: t=10.746, p=0.0000
EM: t=nan, p=nan


### Ragas AA

In [2]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [3]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_gpt4o_GaRAGe_UND_Gemini_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_gpt4o_GaRAGe_UND_Gemini_all_new_scores.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  7.28ba/s]


2001313

In [4]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 50.08
modified AA (avg): 63.60
AA: t=5.441, p=0.0000


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_gpt4o_GaRAGe_UND_Gemini_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.97s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 603
Generation complete: 603 prompts
Average prompt length: 798 bytes (~199 tokens)

Analyze the following input user query:

{"query": "What specific strategies can U.S.-based businesses across various industries employ to address implementation, reporting, and transitional adjustment challenges associated with ASC 842 compliance, particularly in areas such as lease classification, measurement, disclosures, and system upgrades?"}

Please provide your analysis in the following JSON format:

{"query": "What specific strategies can U.S.-based businesses across various industries employ to address implementation, reporting, and transitional adjustment challenges associated with ASC 842 compliance, particularly in areas such as lease classification, measurement, disclosures, and system upgrades?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 121/121 [1:15:34<00:00, 37.47s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,What strategies can businesses employ to mitig...,What specific strategies can U.S.-based busine...,"['To mitigate ASC 842 compliance challenges, b...",['* Start early with planning and assessment...,The query seeks strategies to address ASC 842 ...,0.169312,0.0,0.50,['U.S.-based businesses can employ the followi...,0,0.257426,0.50,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What specific strategies can U....",fully specified
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Convolutional Neural Network...,"[""The Deep Retinal Convolutional Neural Networ...",['* The Deep Retinal Convolution Neural Netw...,The query references 'Deep Retinal Convolution...,0.193548,0.0,0.00,"[""Here's how the DCNN approach improves speech...",0,0.378698,0.25,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How does the Deep Convolutional...",fully specified
2,"How does Zoe Law's ""Legends"" exhibition reflec...","How does Zoë Law's ""Legends"" exhibition use in...","['The ""Legends"" exhibition by Zoë Law reflects...",['* Empowerment of women through reimagined ...,The query specifies a particular artist (Zoe L...,0.178862,0.0,0.00,['* By featuring influential figures from ar...,0,0.317757,1.00,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Zoë Law's \""Legends\"" ...",fully specified
3,How does the involvement of Tyco Ventures and ...,How does the $25 million investment by Tyco Ve...,"[""The involvement of Tyco Ventures and Integra...","[""* **Technological Prospects:**\n * Ty...",The query identifies specific entities (Tyco V...,0.321678,0.0,0.75,"[""* Accelerates R&D and product development ...",0,0.327869,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How does the $25 million invest...",fully specified
4,How has the Drake-Kendrick Lamar feud influenc...,How has the ongoing feud between Drake and Ken...,['The feud between Drake and Kendrick Lamar ha...,"[""* Re-emphasis on lyrical prowess and battl...",The query identifies a specific subject (the D...,0.190476,0.0,1.00,['* **Artistic Competition:** It significant...,0,0.242424,1.00,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How has the ongoing feud betwee...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,How did Elon Musk's opposition to the proposed...,['Elon Musk\'s opposition led to the president...,"[""There is no widely reported instance of Elon...",The query lacks critical specificity. It refer...,0.113821,0.0,0.00,"[""There is no publicly available information i...",0,0.254417,0.00,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Elon Musk's oppositio...",fully specified
599,What challenges does self-managed OpenSearch d...,What are the key challenges faced in self-mana...,['Self-managed OpenSearch deployments face sev...,['* Complex setup and configuration\n* Ong...,The query asks about challenges associated wit...,0.129870,0.0,1.00,['* **Infrastructure Management:**\n * ...,0,0.339506,1.00,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What are the key challenges fac...",fully specified
600,What are the implications of Cleveland-Cliffs ...,What are the potential implications of Clevela...,"[""Cleveland-Cliffs CEO's plan to make another ...","[""* Prolonged uncertainty for U.S. Steel's f...",The query specifies key elements: the subject ...,0.125000,0.0,0.75,['* Increased consolidation of the domestic ...,0,0.194175,0.75,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What are the potential implicat...",fully specified
601,How has the expansion of 

In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.746269
underspecified     0.253731
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    450
underspecified     153
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/GaRAGe_UND_gpt4o_rewritten_reclassified.csv')